# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the **FAIR² Mental Health Survey Dataset from Kilifi County, Kenya** using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and conforms to the [MLCommons Croissant format](http://mlcommons.org/croissant/1.0).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"Identifier: {metadata.identifier}")
print(f"Keywords: {', '.join(metadata.keywords)}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll examine the dataset's structure and print available record sets, as well as their fields and columns. All entities are referenced by their `@id`, as required by the Croissant specification.

In [ ]:
# Display available record sets and their fields/columns by @id

record_sets = list(dataset.record_sets)
print("Available record sets and their fields:")
for rs in record_sets:
    print(f"\nRecordSet: {rs['@id']}")
    fields = rs.get('field', []) if isinstance(rs.get('field', []), list) else [rs.get('field')] if rs.get('field') else []
    print(f"  Fields:")
    for field in fields:
        if isinstance(field, dict) and '@id' in field:
            print(f"    - {field['@id']}")
        elif isinstance(field, str):
            print(f"    - {field}")
    columns = rs.get('column', []) if isinstance(rs.get('column', []), list) else [rs.get('column')] if rs.get('column') else []
    if columns:
        print(f"  Columns:")
        for column in columns:
            if isinstance(column, dict) and '@id' in column:
                print(f"    - {column['@id']}")
            elif isinstance(column, str):
                print(f"    - {column}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record sets and fields are referenced by their `@id`.

Below we list all record set IDs, then select a primary survey table for demonstration.

In [ ]:
# List all record set @ids
record_sets_ids = [rs['@id'] for rs in record_sets]
print(f"All record set @ids: {record_sets_ids}")

# For demonstration, pick the first available record set (typically, primary survey data)
main_record_set_id = record_sets_ids[0] if len(record_sets_ids) > 0 else None
if main_record_set_id is None:
    raise RuntimeError("No record sets found in this dataset.")

# Extract records from all record sets and store in DataFrames
dataframes = {}
for rs_id in record_sets_ids:
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set {rs_id}")
    else:
        print(f"No records found for record set {rs_id}")

df = dataframes[main_record_set_id]
print(f"\nColumns in primary record set ({main_record_set_id}):\n{df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing fields, and grouping data. All operations reference fields by their `@id`.

We'll filter by a numeric field (e.g., total score of the PHQ-9 depression scale) and group by a demographic variable (e.g., gender or education level) if present.

In [ ]:
# Update these values based on the data overview section:
# Suppose PHQ-9 total score field @id is 'phq9_total_score', and gender is 'gender'.
# Use the real @ids from the data overview code cell above.

from pandas.api.types import is_numeric_dtype

# Try to find a likely numeric score field @id in df.columns
possible_score_fields = [col for col in df.columns if 'phq' in col.lower() or 'score' in col.lower()]
numeric_field = None
for col in possible_score_fields:
    if is_numeric_dtype(df[col]):
        numeric_field = col
        break
if not numeric_field:
    # Fallback: try any numeric field
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            numeric_field = col
            break

if not numeric_field:
    raise RuntimeError("No numeric field found for filtering/EDA.")

print(f"Selected numeric_field: {numeric_field}")

# Choose a threshold for demonstration (e.g., PHQ-9 moderate depression)
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"\nRecords where '{numeric_field}' > {threshold}:")
print(filtered_df[[numeric_field]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}':")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt to group by a demographic variable, e.g., 'gender' or 'education'
possible_group_fields = [col for col in df.columns if 'gender' in col.lower() or 'educ' in col.lower() or 'age' in col.lower()]
group_field = possible_group_fields[0] if len(possible_group_fields) > 0 else None

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by '{group_field}' (showing group means):")
    print(grouped_df.head())
else:
    print("No suitable group field found for aggregation.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll create a histogram of the selected numeric field and a boxplot if a group field is present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot by group field, if available
if group_field is not None:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()
else:
    print("No group field available for boxplot visualization.")

## 6. Conclusion
In this notebook, we loaded a FAIR² mental health survey dataset from Kilifi County, Kenya by referencing entities via their Croissant `@id` fields, inspected record sets and fields, performed data extraction, filtering, normalization, grouping, and basic visualization. Results showcased how to quickly explore and analyze complex, schema-based datasets with `mlcroissant`. Continue your analysis by repeating the process for other record sets or fields as needed.